# 02. Data Preparation

This notebook prepares the dataset for supervised churn modeling.

The goal of this step is to:
- define valid feature representations
- prevent data leakage
- split data into train and test sets before fitting any transformations
- build a reusable preprocessing pipeline

## 2.1. Load data

In [2]:
import pandas as pd

df = pd.read_csv("../data/saas_churn.csv")
df.head()

,customer_id,plan,region,industry,tenure_months,team_size,monthly_price,logins_per_week,campaigns_per_month,automation_used,onboarding_completed,integrations_connected,support_tickets_90d,avg_response_hours,nps,failed_payments_6m,discount_used,churn
0,1,Business,EU,SaaS,25,1,94.16,4.58,6,0,1,0,1,13.4,18.0,1,0,0
1,2,Basic,NaN,SaaS,7,19,19.58,1.13,4,0,1,1,0,NaN,63.0,0,0,1
2,3,Business,EU,E-commerce,18,4,105.14,2.48,4,0,1,2,1,6.0,35.0,0,0,0
3,4,Pro,NaN,Fintech,39,3,32.05,4.58,7,1,1,3,0,NaN,6.0,0,0,1
4,5,Basic,NaN,E-commerce,30,15,23.40,5.77,4,0,1,3,2,16.8,NaN,0,1,0


In [3]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             6000 non-null   int64  
 1   plan                    6000 non-null   object 
 2   region                  3313 non-null   object 
 3   industry                6000 non-null   object 
 4   tenure_months           6000 non-null   int64  
 5   team_size               6000 non-null   int64  
 6   monthly_price           5864 non-null   float64
 7   logins_per_week         6000 non-null   float64
 8   campaigns_per_month     6000 non-null   int64  
 9   automation_used         6000 non-null   int64  
 10  onboarding_completed    6000 non-null   int64  
 11  integrations_connected  6000 non-null   int64  
 12  support_tickets_90d     6000 non-null   int64  
 13  avg_response_hours      4161 non-null   float64
 14  nps                     4491 non-null   

## 2.2. Initial sanity checks

In [5]:
df.isna().sum().sort_values(ascending=False)
df["churn"].value_counts(normalize=True)

churn
0    0.662
1    0.338
Name: proportion, dtype: float64

## 2.3. Leakage check
All features must be available at the moment when the churn prediction is made.

Variables that are observed after churn or that directly encode the outcome must not be used as predictors.

## 2.4. Define preprocessing rules

### 2.4.1 Feature types
Numeric features:
- tenure_months
- team_size
- monthly_price
- logins_per_week
- campaigns_per_month
- integrations_connected
- support_tickets_90d
- avg_response_hours
- nps
- failed_payments_6m

Categorical features:
- plan
- region
- industry

Binary features:
- automation_used
- onboarding_completed
- discount_used

### 2.4.2. Missing value strategy
**Numeric features:** imputed using median (robust to outliers)

**Categorical features:** imputed with explicit "Unknown" category

**Binary features:** missing values treated as 0 unless business logic suggests otherwise

### 2.4.3. Encoding
Categorical features will be encoded using One-Hot Encoding.
This allows models to learn non-ordinal category effects.

## 2.5. Split

### 2.5.1. Split features and target

In [6]:
target_col = "churn"

y = df[target_col].copy()
X = df.drop(columns=[target_col]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target in X?", target_col in X.columns)
print("y value counts:\n", y.value_counts(normalize=True).round(3))

X shape: (6000, 17)
y shape: (6000,)
Target in X? False
y value counts:
 churn
0    0.662
1    0.338
Name: proportion, dtype: float64


In [7]:
y.head()

0    0
1    1
2    0
3    1
4    0
Name: churn, dtype: int64

In [8]:
X.head()

,customer_id,plan,region,industry,tenure_months,team_size,monthly_price,logins_per_week,campaigns_per_month,automation_used,onboarding_completed,integrations_connected,support_tickets_90d,avg_response_hours,nps,failed_payments_6m,discount_used
0,1,Business,EU,SaaS,25,1,94.16,4.58,6,0,1,0,1,13.4,18.0,1,0
1,2,Basic,NaN,SaaS,7,19,19.58,1.13,4,0,1,1,0,NaN,63.0,0,0
2,3,Business,EU,E-commerce,18,4,105.14,2.48,4,0,1,2,1,6.0,35.0,0,0
3,4,Pro,NaN,Fintech,39,3,32.05,4.58,7,1,1,3,0,NaN,6.0,0,0
4,5,Basic,NaN,E-commerce,30,15,23.40,5.77,4,0,1,3,2,16.8,NaN,0,1


### 2.5.2. Train / Test split

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True).round(3))

X_train shape: (4800, 17)
X_test shape: (1200, 17)

y_train distribution:
churn
0    0.662
1    0.338
Name: proportion, dtype: float64

y_test distribution:
churn
0    0.662
1    0.338
Name: proportion, dtype: float64


## 2.6. Save Artifacts

In [13]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)

y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)